# Creating LLM

In [13]:
#reading the verdict from verdict.txt file
with open('verdict.txt', 'r') as file:
    verdict = file.read().strip()
print("The length of the verdict is:", len(verdict))
print(verdict[:100])

The length of the verdict is: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [14]:
# Using tiktoken library to tokenize the verdict (BPE tokenization)
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(verdict)
print(encoded[:10])

#decoding the token ids back to text
decoded_ids = tokenizer.decode(encoded)
print(decoded_ids[:100])

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [15]:
# Implementing Data Sampling
context_size = 4
x = encoded[:context_size]
y = encoded[1:context_size+1]
print("X",x)
print("Y",y)

X [40, 367, 2885, 1464]
Y [367, 2885, 1464, 1807]


In [16]:
for i in range(1,context_size+1,2):
    context = encoded[:i]
    target = encoded[:i+1]
    print(f"{tokenizer.decode(context)} ----> {tokenizer.decode(target)}")

I ----> I H
I HAD ----> I HAD always


In [17]:
import torch 
from torch import nn
from torch.utils.data import Dataset, DataLoader

class GPTDataset(Dataset):
    def __init__(self, encoded_data, tokenizer, max_length, stride):
        self.input_id = []
        self.target_id = []

        for i in range(0, len(encoded_data)-max_length, stride):
            input_chunk = encoded_data[i:i+max_length]
            target_chunk = encoded_data[i+1:i+max_length+1]
            self.input_id.append(torch.tensor(input_chunk))
            self.target_id.append(torch.tensor(target_chunk))
    def __len__(self):
        return len(self.input_id)
    
    def __getitem__(self, idx):
        return self.input_id[idx], self.target_id[idx]

In [18]:
# dataset = GPTDataset(encoded, tokenizer, max_length=10, stride=4)
# dataset.__len__()
# dataset.__getitem__(0)

In [19]:
def create_dataloader(txt,batch_size=8, max_length=4, stride=4,shuffle=False,drop_last = True, num_workers=0):
    tokenizer = tiktoken.get_encoding("gpt2")
    data = GPTDataset(tokenizer.encode(txt), tokenizer, max_length, stride)
    dataLoader = DataLoader(data, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)
    return dataLoader

In [20]:
dataLoader = create_dataloader(verdict,batch_size=8,max_length=4,stride=4,shuffle=False)
# data_itr = iter(dataLoader)
# inputs, targets = next(data_itr)
# print(inputs)
# print(inputs.shape,targets.shape)

#Output :
# tensor([[   40,   367,  2885,  1464],
#         [ 1807,  3619,   402,   271],
#         [10899,  2138,   257,  7026],
#         [15632,   438,  2016,   257],
#         [  922,  5891,  1576,   438],
#         [  568,   340,   373,   645],
#         [ 1049,  5975,   284,   502],
#         [  284,  3285,   326,    11]])
# torch.Size([8, 4]) torch.Size([8, 4])

In [21]:
# Embedding the input tokens int the form of 8*4*256
token_encodingLayer = nn.Embedding(50257,256)

# token_encoding = token_encodingLayer(inputs)
# print(token_encoding.shape)

# Output : torch.Size([8, 4, 256])

In [22]:
#Creating positional encodings for the input tokens
pos_embeddingLayer = nn.Embedding(4,256)
pos_encodings = pos_embeddingLayer(torch.arange(4))
print(pos_encodings.shape,pos_encodings)

torch.Size([4, 256]) tensor([[-0.0233,  0.4394, -1.2574,  ..., -0.5265,  1.6807,  1.2768],
        [ 0.4893,  0.3779, -0.1063,  ..., -1.2898, -2.9390,  0.5065],
        [-0.1676,  0.5532, -2.3852,  ...,  0.2160, -1.3454, -1.0386],
        [ 0.0441,  1.0897, -1.4015,  ...,  1.0778,  0.6350, -1.4008]],
       grad_fn=<EmbeddingBackward0>)


In [23]:
# Pocessing each batch
for batch_num, (inputs, targets) in enumerate(dataLoader):
    token_encoding = token_encodingLayer(inputs)
    input_embeddings = token_encoding + pos_encodings
    # print(f"Batch {batch_num}: {token_encoding.shape}")

In [24]:
Wq = nn.Parameter(torch.randn(3,2), requires_grad=True)
x = torch.tensor([0.2, 0., 0.5])    # Shape: [3, 1]
print(x @ Wq)

tensor([-0.4576, -0.5481], grad_fn=<SqueezeBackward4>)


## Implementing Casual Attention Class

In [25]:
class CasualSelfAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=True):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        # register_buffer is similar to nn.prameter but it is not a lernable parameter, it is a constant that is saved with the model
        self.register_buffer("mask", torch.triu(torch.ones(context_length,context_length),diagonal=1))
    
    def forward(self,x):
        b, num_token, d_in = x.shape
        keys = self.W_key(x)
        query = self.W_query(x)
        value = self.W_value(x)

        attn_score = query @ keys.transpose(1,2)
        attn_score = attn_score.masked_fill(self.mask[:num_token,:num_token] == 1, float("-inf"))
        attn_weights = torch.softmax(attn_score / keys.shape[-1]**0.5, dim=-1)

        attn_weights = self.dropout(attn_weights)
        context_vec = attn_weights @ value
        return context_vec

In [ ]:
context_length = 4
ca = CasualSelfAttention(d_in=256, d_out=2, context_length=context_length, dropout=0.1)
context_vec = ca(input_embeddings)
print(context_vec.shape)

print('''
This is one of the batch which has 8 sentences with 4 tokens and each token is represented by
256 dimensional vector. After applying the casual self attention we get the context vector of each token
of shape [1,2]. This is for one token, we have 4 tokens in each sentence and 8 sentences in a batch. 
So the output shape is [8,4,2].
''')

torch.Size([8, 4, 2])

This is one of the batch which has 8 sentences with 4 tokens and each token is represented by
256 dimensional vector. After applying the casual self attention we get the context vector of each token
of shape [1,2]. This is for one token, we have 4 tokens in each sentence and 8 sentences in a batch. 
So the output shape is [8,4,2].



## Wrapper class to implement Multi-head attention 

In [36]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.head = nn.ModuleList([
            CasualSelfAttention(d_in=d_in, d_out=d_out, context_length=context_length, dropout=0.1) for _ in range(num_heads)
        ])
    
    def forward(self,x):
        return torch.cat([head(x) for head in self.head], dim=1)

In [38]:
mha = MultiHeadAttention(d_in=256, d_out=2, context_length=context_length, dropout=0.1, num_heads=2)
context_v = mha(input_embeddings)
print(context_v.shape)
print(context_v)

torch.Size([8, 8, 2])
tensor([[[-1.8215, -0.9050],
         [-1.4285, -0.5233],
         [-0.7130, -0.2625],
         [-0.7475,  0.0393],
         [-0.0439,  0.0869],
         [-0.0363,  0.0719],
         [ 0.7942, -0.1883],
         [ 0.4195, -0.3482]],

        [[ 0.0000,  0.0000],
         [-0.4181, -0.4244],
         [-1.1697,  0.1266],
         [-1.0758,  0.1915],
         [-0.4250, -0.7130],
         [-0.1535, -0.7593],
         [ 0.4002, -0.7018],
         [ 0.2220, -0.4887]],

        [[-1.9670, -0.7347],
         [-1.6666,  0.1320],
         [-1.0688,  0.1660],
         [-0.8431,  0.2499],
         [ 0.3513, -0.5996],
         [ 0.3284, -0.5526],
         [ 0.4437,  0.7232],
         [ 0.4033, -0.1555]],

        [[-0.9916, -0.9411],
         [-0.8379,  0.0445],
         [-0.1389, -0.0038],
         [-0.9090,  0.0807],
         [-1.0709,  0.2742],
         [-0.9497,  0.2612],
         [ 0.7695,  0.0991],
         [-0.3389, -0.1099]],

        [[-0.4936,  0.5046],
         [-0.